# DSA Text Analysis — Correlations & Panel Construction\n\n**Inputs:**\n- `dsa_parent_pairs.xlsx` — DSA ↔ parent document mapping\n- `mentions_all_doctypes.xlsx` — hitcount data for all documents\n- `imf_pgm_dummy_data_v2.xlsx` — IMF program dummy by country-month\n\n**Outputs:**\n1. `panel_country_year.xlsx` — Time-series panel (67 countries × 20 years)\n2. `correlation_by_topic.xlsx` — Aggregate correlation DSA↔parent per topic\n3. `alignment_scores.xlsx` — Per-pair alignment score\n4. `comparison_program_vs_no_program.xlsx` — Program vs non-program analysis

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
sns.set_style('whitegrid')
print("Libraries loaded.")

In [ ]:
# Load the three data sources
pairs = pd.read_excel('inputs/dsa_parent_pairs.xlsx', sheet_name='Sheet1')
mentions = pd.read_excel('inputs/mentions_all_doctypes.xlsx', sheet_name='Raw Data')
imf_pgm = pd.read_excel('inputs/imf_pgm_dummy_data_v2.xlsx', sheet_name='Main_Data')

print(f"Pairs:    {pairs.shape[0]} rows, {pairs.shape[1]} cols")
print(f"Mentions: {mentions.shape[0]} rows, {mentions.shape[1]} cols")
print(f"IMF Pgm:  {imf_pgm.shape[0]} rows, {imf_pgm.shape[1]} cols")

In [ ]:
# Identify topic columns from mentions (exclude paired_* columns)
ignore_cols = {'paired_with', 'paired_filename', 'paired_conditional', 'paired_conditional_filename'}
meta_cols = {'filename', 'country', 'year', 'month', 'ifs', 'doc_type', 'region',
             'arrangement_number', 'review', 'word_count',
             'cex', 'fcs', 'sds', 'rst', 'hipc', 'frontier_market', 'exposure_china'}

# Extract topic names from the has_* columns
topic_names = [c.replace('has_', '') for c in mentions.columns
               if c.startswith('has_') and c not in ignore_cols]

# Build column lists for each metric type
has_cols = [f'has_{t}' for t in topic_names]
count_cols = [f'count_{t}' for t in topic_names]
per100w_cols = [f'count_{t}_per100w' for t in topic_names]
topic_data_cols = has_cols + count_cols + per100w_cols

print(f"Topics found: {len(topic_names)}")
print(f"Sample topics: {topic_names[:10]}")
print(f"Topic data columns: {len(topic_data_cols)}")

## 2. Data Cleaning & Linking\n\nFilter out parentless DSAs, link DSAs and parents to their mention data.

In [ ]:
# Clean pairs: drop unnamed columns, fill parentless NaN as 0
pairs = pairs.loc[:, ~pairs.columns.isna()]
pairs['parentless'] = pairs['parentless'].fillna(0).astype(int)

# Split into valid pairs and parentless
valid_pairs = pairs[pairs['parentless'] == 0].copy()
parentless_dsas = pairs[pairs['parentless'] == 1].copy()

# Also exclude the 2 DSAs whose parent is not in mentions (Mauritania, Sao Tome)
missing_parents = {'682_764_R5.md', '716_786_R1.md'}
valid_pairs = valid_pairs[~valid_pairs['parent_filename'].isin(missing_parents)].copy()

print(f"Total DSAs: {len(pairs)}")
print(f"Parentless: {len(parentless_dsas)}")
print(f"Missing parent in mentions: {len(pairs[pairs['parent_filename'].isin(missing_parents)])}")
print(f"Valid pairs for analysis: {len(valid_pairs)}")

In [ ]:
# Create mentions lookup by filename
mentions_lookup = mentions.set_index('filename')

# Merge DSA topic data onto valid_pairs
dsa_topic_data = mentions_lookup.loc[
    mentions_lookup.index.isin(valid_pairs['filename']),
    ['word_count'] + topic_data_cols
]
dsa_topic_data = dsa_topic_data.rename(
    columns={c: f'{c}_dsa' for c in ['word_count'] + topic_data_cols}
)

# Merge parent topic data onto valid_pairs
parent_topic_data = mentions_lookup.loc[
    mentions_lookup.index.isin(valid_pairs['parent_filename']),
    ['word_count'] + topic_data_cols
]
parent_topic_data = parent_topic_data.rename(
    columns={c: f'{c}_parent' for c in ['word_count'] + topic_data_cols}
)

# Build the master working dataframe
df = valid_pairs.merge(dsa_topic_data, left_on='filename', right_index=True, how='left')
df = df.merge(parent_topic_data, left_on='parent_filename', right_index=True, how='left')

print(f"Working dataframe: {df.shape[0]} rows, {df.shape[1]} cols")
print(f"\\nColumns sample: {list(df.columns[:15])}...")
df[['filename', 'parent_filename', 'parent_type', 'during_program',
    'word_count_dsa', 'word_count_parent']].head()

## 3. Output 1 — Panel País-Año\n\n67 countries × 20 years (2005–2024). When multiple DSAs exist for a country-year, keep the latest one (highest month). Years without a DSA have empty topic columns (ready for IDS merge).

In [ ]:
# Build the full country-year skeleton using ALL DSAs (including parentless)
all_countries = pairs[['country', 'ifs']].drop_duplicates()
all_years = pd.DataFrame({'year': range(2005, 2025)})
skeleton = all_countries.merge(all_years, how='cross')
print(f"Panel skeleton: {len(skeleton)} rows ({len(all_countries)} countries × {len(all_years)} years)")

# For DSA selection: use valid_pairs (with parent) — pick last DSA per country-year
df_sorted = df.sort_values('month', ascending=False)
last_dsa = df_sorted.groupby(['country', 'year']).first().reset_index()

# Merge onto skeleton
panel = skeleton.merge(last_dsa, on=['country', 'year'], how='left', suffixes=('', '_drop'))

# Clean up: use ifs from skeleton, drop duplicates
panel['ifs'] = panel['ifs'].combine_first(panel.get('ifs_drop', panel['ifs']))
drop_cols = [c for c in panel.columns if c.endswith('_drop')]
panel = panel.drop(columns=drop_cols)

# Add a flag for whether this row has a DSA
panel['has_dsa'] = panel['filename'].notna().astype(int)

# Reorder columns: identifiers first, then meta, then DSA topics, then parent topics
id_cols = ['country', 'ifs', 'year', 'has_dsa', 'filename', 'month',
           'parent_filename', 'parent_type', 'during_program',
           'cex', 'fcs', 'sds', 'hipc', 'frontier_market', 'exposure_china']
dsa_data = ['word_count_dsa'] + [f'{c}_dsa' for c in topic_data_cols]
parent_data = ['word_count_parent'] + [f'{c}_parent' for c in topic_data_cols]

# Only keep columns that exist
id_cols = [c for c in id_cols if c in panel.columns]
dsa_data = [c for c in dsa_data if c in panel.columns]
parent_data = [c for c in parent_data if c in panel.columns]

panel = panel[id_cols + dsa_data + parent_data]
panel = panel.sort_values(['country', 'year']).reset_index(drop=True)

print(f"\\nPanel: {panel.shape[0]} rows, {panel.shape[1]} cols")
print(f"Rows with DSA: {panel['has_dsa'].sum()}")
print(f"Rows without DSA: {(panel['has_dsa'] == 0).sum()}")
panel.head(10)

In [ ]:
# Save panel
panel.to_excel('outputs/correlations/panel_country_year.xlsx', index=False, sheet_name='Panel')
print("Saved: panel_country_year.xlsx")

## 4. Output 2a — Correlation by Topic (Aggregate)\n\nFor each topic, correlate `count_topic_per100w` in the DSA vs in the parent across all valid pairs.\nThis answers: "When a staff report talks a lot about topic X, does the DSA also talk about it?"

In [ ]:
# Compute correlation for each topic: DSA per100w vs Parent per100w
corr_results = []

for topic in topic_names:
    dsa_col = f'count_{topic}_per100w_dsa'
    parent_col = f'count_{topic}_per100w_parent'

    # Drop NaN pairs
    mask = df[dsa_col].notna() & df[parent_col].notna()
    x = df.loc[mask, dsa_col]
    y = df.loc[mask, parent_col]
    n = len(x)

    if n < 3:
        corr_results.append({
            'topic': topic, 'n_pairs': n,
            'corr_pearson': np.nan, 'p_pearson': np.nan,
            'corr_spearman': np.nan, 'p_spearman': np.nan,
            'mean_per100w_dsa': np.nan, 'mean_per100w_parent': np.nan
        })
        continue

    r_p, p_p = stats.pearsonr(x, y)
    r_s, p_s = stats.spearmanr(x, y)

    corr_results.append({
        'topic': topic,
        'n_pairs': n,
        'corr_pearson': round(r_p, 4),
        'p_pearson': round(p_p, 6),
        'corr_spearman': round(r_s, 4),
        'p_spearman': round(p_s, 6),
        'mean_per100w_dsa': round(x.mean(), 4),
        'mean_per100w_parent': round(y.mean(), 4),
    })

corr_by_topic = pd.DataFrame(corr_results).sort_values('corr_spearman', ascending=False)
corr_by_topic = corr_by_topic.reset_index(drop=True)

print(f"Topics with significant Spearman correlation (p < 0.05): "
      f"{(corr_by_topic['p_spearman'] < 0.05).sum()} / {len(corr_by_topic)}")
corr_by_topic.head(15)

In [ ]:
# Visualize: top and bottom correlated topics (Spearman)
fig, ax = plt.subplots(figsize=(10, 12))
data_plot = corr_by_topic.dropna(subset=['corr_spearman']).copy()
colors = ['#2ecc71' if p < 0.05 else '#95a5a6' for p in data_plot['p_spearman']]
ax.barh(range(len(data_plot)), data_plot['corr_spearman'], color=colors)
ax.set_yticks(range(len(data_plot)))
ax.set_yticklabels(data_plot['topic'], fontsize=8)
ax.set_xlabel('Spearman Correlation (DSA vs Parent)')
ax.set_title('Topic-level Correlation: DSA ↔ Parent Document')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color='#2ecc71', label='p < 0.05'),
    plt.Rectangle((0,0),1,1, color='#95a5a6', label='p ≥ 0.05')
], loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Save
corr_by_topic.to_excel('outputs/correlations/correlation_by_topic.xlsx', index=False, sheet_name='By Topic')
print("Saved: correlation_by_topic.xlsx")

## 5. Output 2b — Alignment Score per Pair\n\nFor each DSA-parent pair, correlate their topic profiles (vector of 68 per100w values).\nThis produces one "alignment score" per pair — how thematically aligned are the two documents?

In [ ]:
# Compute alignment score for each pair
dsa_per100w = [f'count_{t}_per100w_dsa' for t in topic_names]
parent_per100w = [f'count_{t}_per100w_parent' for t in topic_names]

alignment_scores = []
for idx, row in df.iterrows():
    dsa_vec = row[dsa_per100w].values.astype(float)
    parent_vec = row[parent_per100w].values.astype(float)

    # Need at least some non-zero variation for correlation
    valid = ~(np.isnan(dsa_vec) | np.isnan(parent_vec))
    dsa_v = dsa_vec[valid]
    par_v = parent_vec[valid]

    if len(dsa_v) < 3 or np.std(dsa_v) == 0 or np.std(par_v) == 0:
        score = np.nan
        p_val = np.nan
    else:
        score, p_val = stats.spearmanr(dsa_v, par_v)

    alignment_scores.append({
        'filename': row['filename'],
        'country': row['country'],
        'ifs': row['ifs'],
        'year': row['year'],
        'month': row['month'],
        'parent_filename': row['parent_filename'],
        'parent_type': row['parent_type'],
        'during_program': row['during_program'],
        'alignment_score': round(score, 4) if not np.isnan(score) else np.nan,
        'alignment_p_value': round(p_val, 6) if not np.isnan(p_val) else np.nan,
    })

align_df = pd.DataFrame(alignment_scores)

print(f"Alignment scores computed: {len(align_df)}")
print(f"Mean alignment: {align_df['alignment_score'].mean():.4f}")
print(f"Median alignment: {align_df['alignment_score'].median():.4f}")
print(f"\\nBy parent_type:")
print(align_df.groupby('parent_type')['alignment_score'].describe().round(3))
align_df.head(10)

In [ ]:
# Visualize alignment score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(align_df['alignment_score'].dropna(), bins=40, edgecolor='black', alpha=0.7)
axes[0].axvline(align_df['alignment_score'].median(), color='red', linestyle='--', label=f"Median: {align_df['alignment_score'].median():.3f}")
axes[0].set_xlabel('Alignment Score (Spearman)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of DSA↔Parent Alignment Scores')
axes[0].legend()

# By parent type
for ptype in ['AIV', 'Program']:
    subset = align_df[align_df['parent_type'] == ptype]['alignment_score'].dropna()
    axes[1].hist(subset, bins=30, alpha=0.5, label=f'{ptype} (n={len(subset)})', edgecolor='black')
axes[1].set_xlabel('Alignment Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Alignment by Parent Type')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Save
align_df.to_excel('outputs/correlations/alignment_scores.xlsx', index=False, sheet_name='Alignment')
print("Saved: alignment_scores.xlsx")

## 6. Output 3 — Program vs No-Program Comparison\n\n### Sheet A (Option A): Topic-level — Do DSAs mention topics differently during programs?\nFor each topic, compare `count_per100w` between DSAs in-program vs not-in-program (Mann-Whitney U test).\n\n### Sheet B (Option B): Pair-level — Does DSA↔Parent alignment change during programs?\nCompare the alignment scores between pairs in-program vs not-in-program.

In [ ]:
# Sheet A: Topic-level comparison — DSA mentions in-program vs not-in-program
in_pgm = df[df['during_program'] == 1]
no_pgm = df[df['during_program'] == 0]

topic_comparison = []
for topic in topic_names:
    col_dsa = f'count_{topic}_per100w_dsa'

    x_in = in_pgm[col_dsa].dropna()
    x_no = no_pgm[col_dsa].dropna()

    if len(x_in) < 2 or len(x_no) < 2:
        continue

    u_stat, p_val = stats.mannwhitneyu(x_in, x_no, alternative='two-sided')

    topic_comparison.append({
        'topic': topic,
        'n_in_program': len(x_in),
        'mean_in_program': round(x_in.mean(), 4),
        'median_in_program': round(x_in.median(), 4),
        'n_not_in_program': len(x_no),
        'mean_not_in_program': round(x_no.mean(), 4),
        'median_not_in_program': round(x_no.median(), 4),
        'diff_means': round(x_in.mean() - x_no.mean(), 4),
        'p_value_mannwhitney': round(p_val, 6),
    })

sheet_a = pd.DataFrame(topic_comparison).sort_values('p_value_mannwhitney')
sheet_a = sheet_a.reset_index(drop=True)

sig = (sheet_a['p_value_mannwhitney'] < 0.05).sum()
print(f"Sheet A: {sig} / {len(sheet_a)} topics with significant difference (p < 0.05)")
print(f"In-program DSAs: {len(in_pgm)}, Not-in-program: {len(no_pgm)}")
sheet_a.head(15)

In [ ]:
# Visualize Sheet A: top significant differences
sig_topics = sheet_a[sheet_a['p_value_mannwhitney'] < 0.05].copy()
if len(sig_topics) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(sig_topics) * 0.4)))
    colors = ['#e74c3c' if d > 0 else '#3498db' for d in sig_topics['diff_means']]
    ax.barh(range(len(sig_topics)), sig_topics['diff_means'], color=colors)
    ax.set_yticks(range(len(sig_topics)))
    ax.set_yticklabels(sig_topics['topic'], fontsize=9)
    ax.set_xlabel('Difference in Mean per100w (In-Program − Not-In-Program)')
    ax.set_title('Significant Topic Differences: DSAs In-Program vs Not-In-Program (p < 0.05)')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.legend(handles=[
        plt.Rectangle((0,0),1,1, color='#e74c3c', label='Higher in program'),
        plt.Rectangle((0,0),1,1, color='#3498db', label='Lower in program')
    ], loc='lower right')
    plt.tight_layout()
    plt.show()
else:
    print("No topics with p < 0.05")

In [ ]:
# Sheet B: Pair-level — alignment score in-program vs not-in-program
align_in = align_df[align_df['during_program'] == 1]['alignment_score'].dropna()
align_no = align_df[align_df['during_program'] == 0]['alignment_score'].dropna()

u_stat, p_val = stats.mannwhitneyu(align_in, align_no, alternative='two-sided')

sheet_b_summary = pd.DataFrame([{
    'metric': 'alignment_score',
    'n_in_program': len(align_in),
    'mean_in_program': round(align_in.mean(), 4),
    'median_in_program': round(align_in.median(), 4),
    'n_not_in_program': len(align_no),
    'mean_not_in_program': round(align_no.mean(), 4),
    'median_not_in_program': round(align_no.median(), 4),
    'diff_means': round(align_in.mean() - align_no.mean(), 4),
    'p_value_mannwhitney': round(p_val, 6),
}])

print("Sheet B: Alignment score comparison")
print(sheet_b_summary.to_string(index=False))

# Also compute per-topic correlation split by program status
pair_corr_comparison = []
for topic in topic_names:
    dsa_col = f'count_{topic}_per100w_dsa'
    parent_col = f'count_{topic}_per100w_parent'

    for label, subset in [('in_program', in_pgm), ('not_in_program', no_pgm)]:
        mask = subset[dsa_col].notna() & subset[parent_col].notna()
        x = subset.loc[mask, dsa_col]
        y = subset.loc[mask, parent_col]
        n = len(x)
        if n < 3:
            r, p = np.nan, np.nan
        else:
            r, p = stats.spearmanr(x, y)
        pair_corr_comparison.append({
            'topic': topic, 'group': label, 'n': n,
            'corr_spearman': round(r, 4) if not np.isnan(r) else np.nan,
            'p_spearman': round(p, 6) if not np.isnan(p) else np.nan,
        })

pair_corr_df = pd.DataFrame(pair_corr_comparison)
pair_corr_pivot = pair_corr_df.pivot(index='topic', columns='group', values='corr_spearman')
pair_corr_pivot['diff_corr'] = pair_corr_pivot['in_program'] - pair_corr_pivot['not_in_program']
pair_corr_pivot = pair_corr_pivot.sort_values('diff_corr', ascending=False).reset_index()

print(f"\\nPer-topic correlation split:")
pair_corr_pivot.head(10)

In [ ]:
# Visualize Sheet B: alignment distribution by program status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Alignment score distributions
axes[0].hist(align_in, bins=30, alpha=0.6, label=f'In-program (n={len(align_in)})', color='#e74c3c', edgecolor='black')
axes[0].hist(align_no, bins=30, alpha=0.6, label=f'Not-in-program (n={len(align_no)})', color='#3498db', edgecolor='black')
axes[0].axvline(align_in.median(), color='#e74c3c', linestyle='--', alpha=0.8)
axes[0].axvline(align_no.median(), color='#3498db', linestyle='--', alpha=0.8)
axes[0].set_xlabel('Alignment Score')
axes[0].set_ylabel('Count')
axes[0].set_title(f'DSA↔Parent Alignment: Program vs No-Program (p={p_val:.4f})')
axes[0].legend()

# Per-topic correlation difference
data_sorted = pair_corr_pivot.dropna().sort_values('diff_corr')
colors = ['#e74c3c' if d > 0 else '#3498db' for d in data_sorted['diff_corr']]
axes[1].barh(range(len(data_sorted)), data_sorted['diff_corr'], color=colors, height=0.7)
axes[1].set_yticks(range(len(data_sorted)))
axes[1].set_yticklabels(data_sorted['topic'], fontsize=6)
axes[1].set_xlabel('Δ Spearman Corr (In-Program − Not-In-Program)')
axes[1].set_title('Per-Topic DSA↔Parent Correlation Difference')
axes[1].axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Save Output 3
with pd.ExcelWriter('outputs/correlations/comparison_program_vs_no_program.xlsx') as writer:
    sheet_a.to_excel(writer, sheet_name='DSA_Only', index=False)
    sheet_b_summary.to_excel(writer, sheet_name='Pair_Alignment_Summary', index=False)
    pair_corr_pivot.to_excel(writer, sheet_name='Pair_Corr_By_Topic', index=False)

print("Saved: comparison_program_vs_no_program.xlsx")
print("  - Sheet 'DSA_Only': topic-level mean comparison (Option A)")
print("  - Sheet 'Pair_Alignment_Summary': overall alignment score comparison (Option B)")
print("  - Sheet 'Pair_Corr_By_Topic': per-topic DSA↔parent correlation split by program status (Option B)")

## 7. Robustness Checks\n\n### 7.1 FDR Correction (Benjamini-Hochberg)\nWith 68 topics tested simultaneously, some p<0.05 results may be false positives.\nFDR controls the expected proportion of false discoveries among rejected hypotheses.

In [ ]:
from statsmodels.stats.multitest import multipletests

# --- FDR on Output 2a: DSA↔Parent correlation p-values ---
pvals_corr = corr_by_topic['p_spearman'].dropna().values
reject_corr, pvals_corr_fdr, _, _ = multipletests(pvals_corr, alpha=0.05, method='fdr_bh')
corr_by_topic_fdr = corr_by_topic.copy()
corr_by_topic_fdr['p_spearman_fdr'] = np.nan
corr_by_topic_fdr.loc[corr_by_topic_fdr['p_spearman'].notna(), 'p_spearman_fdr'] = pvals_corr_fdr
corr_by_topic_fdr['sig_after_fdr'] = corr_by_topic_fdr['p_spearman_fdr'] < 0.05

orig_sig = (corr_by_topic['p_spearman'] < 0.05).sum()
fdr_sig = corr_by_topic_fdr['sig_after_fdr'].sum()
print(f"Output 2a (DSA↔Parent correlation):")
print(f"  Significant before FDR: {orig_sig}/68")
print(f"  Significant after FDR:  {fdr_sig}/68")

# --- FDR on Output 3 Sheet A: Program vs No-Program ---
pvals_pgm = sheet_a['p_value_mannwhitney'].values
reject_pgm, pvals_pgm_fdr, _, _ = multipletests(pvals_pgm, alpha=0.05, method='fdr_bh')
sheet_a_fdr = sheet_a.copy()
sheet_a_fdr['p_mannwhitney_fdr'] = pvals_pgm_fdr
sheet_a_fdr['sig_after_fdr'] = sheet_a_fdr['p_mannwhitney_fdr'] < 0.05

orig_sig_a = (sheet_a['p_value_mannwhitney'] < 0.05).sum()
fdr_sig_a = sheet_a_fdr['sig_after_fdr'].sum()
print(f"\nOutput 3 Sheet A (Program vs No-Program, DSA only):")
print(f"  Significant before FDR: {orig_sig_a}/68")
print(f"  Significant after FDR:  {fdr_sig_a}/68")

# Show topics that lost significance
lost = sheet_a_fdr[(sheet_a_fdr['p_value_mannwhitney'] < 0.05) & (~sheet_a_fdr['sig_after_fdr'])]
if len(lost) > 0:
    print(f"\n  Topics that lost significance after FDR ({len(lost)}):")
    for _, r in lost.iterrows():
        print(f"    {r['topic']}: p={r['p_value_mannwhitney']:.4f} → p_fdr={r['p_mannwhitney_fdr']:.4f}")

### 7.2 Outlier Check (Winsorization)\nWinsorize `count_per100w` at the 1st and 99th percentile to check whether extreme values are driving correlations.

In [ ]:
from scipy.stats.mstats import winsorize

# Re-run Output 2a correlations with winsorized data
corr_winsor = []
for topic in topic_names:
    dsa_col = f'count_{topic}_per100w_dsa'
    parent_col = f'count_{topic}_per100w_parent'

    mask = df[dsa_col].notna() & df[parent_col].notna()
    x = df.loc[mask, dsa_col].values
    y = df.loc[mask, parent_col].values
    n = len(x)

    if n < 3:
        corr_winsor.append({'topic': topic, 'corr_original': np.nan, 'corr_winsorized': np.nan})
        continue

    # Original
    r_orig, _ = stats.spearmanr(x, y)
    # Winsorized at 1%/99%
    x_w = winsorize(x, limits=[0.01, 0.01])
    y_w = winsorize(y, limits=[0.01, 0.01])
    r_win, _ = stats.spearmanr(x_w, y_w)

    corr_winsor.append({
        'topic': topic,
        'corr_original': round(r_orig, 4),
        'corr_winsorized': round(r_win, 4),
        'diff': round(r_win - r_orig, 4),
    })

winsor_df = pd.DataFrame(corr_winsor).dropna()
winsor_df['abs_diff'] = winsor_df['diff'].abs()

print("Winsorization impact on Spearman correlations (Output 2a):")
print(f"  Mean absolute change: {winsor_df['abs_diff'].mean():.4f}")
print(f"  Max absolute change:  {winsor_df['abs_diff'].max():.4f}")
print(f"\nTopics most affected by winsorization:")
print(winsor_df.sort_values('abs_diff', ascending=False).head(10).to_string(index=False))

# Scatter: original vs winsorized
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(winsor_df['corr_original'], winsor_df['corr_winsorized'], alpha=0.6, edgecolor='black')
ax.plot([-0.2, 0.8], [-0.2, 0.8], 'r--', alpha=0.5, label='y=x (no change)')
ax.set_xlabel('Original Spearman Correlation')
ax.set_ylabel('Winsorized Spearman Correlation')
ax.set_title('Robustness: Effect of Winsorization on Correlations')
ax.legend()
plt.tight_layout()
plt.show()

### 7.3 Country Concentration Check\nAre results driven by a few countries? Leave-one-country-out: for each of the top 5 most significant topics, re-run the correlation dropping one country at a time and check stability.

In [ ]:
# Leave-one-country-out for top correlated topics AND top program-vs-no-program topics
countries_list = df['country'].unique()

# Pick top 5 topics from Output 2a (highest absolute Spearman)
top5_corr = corr_by_topic.dropna().head(5)['topic'].tolist()

# Pick top 5 topics from Output 3 Sheet A (lowest p-value)
top5_pgm = sheet_a.head(5)['topic'].tolist()

# Combine unique topics
check_topics = list(dict.fromkeys(top5_corr + top5_pgm))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left panel: Leave-one-out for DSA↔Parent correlation ---
loo_corr_data = {}
for topic in top5_corr:
    dsa_col = f'count_{topic}_per100w_dsa'
    parent_col = f'count_{topic}_per100w_parent'
    loo_vals = []
    for c in countries_list:
        subset = df[df['country'] != c]
        mask = subset[dsa_col].notna() & subset[parent_col].notna()
        x = subset.loc[mask, dsa_col]
        y = subset.loc[mask, parent_col]
        if len(x) >= 3:
            r, _ = stats.spearmanr(x, y)
            loo_vals.append(r)
    loo_corr_data[topic] = loo_vals

bp1 = axes[0].boxplot([loo_corr_data[t] for t in top5_corr], vert=True, patch_artist=True)
axes[0].set_xticklabels(top5_corr, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Spearman Correlation')
axes[0].set_title('Leave-One-Country-Out: DSA↔Parent Correlation\n(Top 5 topics)')
# Add original values as red dots
for i, topic in enumerate(top5_corr):
    orig_val = corr_by_topic[corr_by_topic['topic'] == topic]['corr_spearman'].values[0]
    axes[0].scatter(i+1, orig_val, color='red', zorder=5, s=50, label='Full sample' if i==0 else '')
axes[0].legend()

# --- Right panel: Leave-one-out for program vs no-program diff ---
loo_pgm_data = {}
for topic in top5_pgm:
    col_dsa = f'count_{topic}_per100w_dsa'
    loo_vals = []
    for c in countries_list:
        subset = df[df['country'] != c]
        x_in = subset[subset['during_program'] == 1][col_dsa].dropna()
        x_no = subset[subset['during_program'] == 0][col_dsa].dropna()
        if len(x_in) >= 2 and len(x_no) >= 2:
            loo_vals.append(x_in.mean() - x_no.mean())
    loo_pgm_data[topic] = loo_vals

bp2 = axes[1].boxplot([loo_pgm_data[t] for t in top5_pgm], vert=True, patch_artist=True)
axes[1].set_xticklabels(top5_pgm, rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('Diff in Means (In-Pgm − Not-In-Pgm)')
axes[1].set_title('Leave-One-Country-Out: Program Effect\n(Top 5 topics)')
axes[1].axhline(y=0, color='grey', linestyle='--', alpha=0.5)
for i, topic in enumerate(top5_pgm):
    orig_val = sheet_a[sheet_a['topic'] == topic]['diff_means'].values[0]
    axes[1].scatter(i+1, orig_val, color='red', zorder=5, s=50, label='Full sample' if i==0 else '')
axes[1].legend()

plt.tight_layout()
plt.show()

# Print stability summary
print("Leave-one-country-out stability (correlation topics):")
for topic in top5_corr:
    vals = loo_corr_data[topic]
    orig = corr_by_topic[corr_by_topic['topic'] == topic]['corr_spearman'].values[0]
    print(f"  {topic}: original={orig:.4f}, LOO range=[{min(vals):.4f}, {max(vals):.4f}]")

### 7.4 Permutation Test for Program vs No-Program\nShuffle the `during_program` label 1,000 times and re-compute the difference in means.\nIf the observed difference falls in the extreme tail of the null distribution, the result is robust.

In [ ]:
np.random.seed(42)
N_PERM = 1000

# Pick top 5 significant topics from Sheet A plus the alignment score
top5_perm = sheet_a.head(5)['topic'].tolist()

# --- Permutation for topic-level differences ---
perm_results = {}
for topic in top5_perm:
    col = f'count_{topic}_per100w_dsa'
    vals = df[col].dropna().values
    labels = df.loc[df[col].notna(), 'during_program'].values

    # Observed difference
    obs_diff = vals[labels == 1].mean() - vals[labels == 0].mean()

    # Permutation null distribution
    null_diffs = []
    for _ in range(N_PERM):
        perm_labels = np.random.permutation(labels)
        null_diffs.append(vals[perm_labels == 1].mean() - vals[perm_labels == 0].mean())

    null_diffs = np.array(null_diffs)
    p_perm = (np.abs(null_diffs) >= np.abs(obs_diff)).mean()
    perm_results[topic] = {'obs_diff': obs_diff, 'null_diffs': null_diffs, 'p_perm': p_perm}

# --- Permutation for alignment score ---
align_vals = align_df['alignment_score'].dropna().values
align_labels = align_df.loc[align_df['alignment_score'].notna(), 'during_program'].values
obs_align_diff = align_vals[align_labels == 1].mean() - align_vals[align_labels == 0].mean()

null_align = []
for _ in range(N_PERM):
    perm_labels = np.random.permutation(align_labels)
    null_align.append(align_vals[perm_labels == 1].mean() - align_vals[perm_labels == 0].mean())
null_align = np.array(null_align)
p_align_perm = (np.abs(null_align) >= np.abs(obs_align_diff)).mean()

# --- Visualize ---
n_plots = len(top5_perm) + 1
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, topic in enumerate(top5_perm):
    r = perm_results[topic]
    axes[i].hist(r['null_diffs'], bins=50, alpha=0.7, color='#95a5a6', edgecolor='black')
    axes[i].axvline(r['obs_diff'], color='red', linewidth=2, label=f"Observed (p_perm={r['p_perm']:.3f})")
    axes[i].set_title(topic, fontsize=10)
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel('Diff in Means')

# Alignment score
axes[len(top5_perm)].hist(null_align, bins=50, alpha=0.7, color='#95a5a6', edgecolor='black')
axes[len(top5_perm)].axvline(obs_align_diff, color='red', linewidth=2,
                              label=f"Observed (p_perm={p_align_perm:.3f})")
axes[len(top5_perm)].set_title('Alignment Score', fontsize=10)
axes[len(top5_perm)].legend(fontsize=8)
axes[len(top5_perm)].set_xlabel('Diff in Means')

plt.suptitle('Permutation Tests: Observed vs Null Distribution (1,000 permutations)', fontsize=13)
plt.tight_layout()
plt.show()

# Summary
print("Permutation test results (two-sided):")
for topic in top5_perm:
    r = perm_results[topic]
    print(f"  {topic}: obs_diff={r['obs_diff']:.4f}, p_perm={r['p_perm']:.3f}")
print(f"  alignment_score: obs_diff={obs_align_diff:.4f}, p_perm={p_align_perm:.3f}")

In [ ]:
# Save robustness-augmented outputs (overwrite originals with FDR columns added)
corr_by_topic_fdr.to_excel('outputs/correlations/correlation_by_topic.xlsx', index=False, sheet_name='By Topic')

with pd.ExcelWriter('outputs/correlations/comparison_program_vs_no_program.xlsx') as writer:
    sheet_a_fdr.to_excel(writer, sheet_name='DSA_Only', index=False)
    sheet_b_summary.to_excel(writer, sheet_name='Pair_Alignment_Summary', index=False)
    pair_corr_pivot.to_excel(writer, sheet_name='Pair_Corr_By_Topic', index=False)
    winsor_df.to_excel(writer, sheet_name='Winsorization_Check', index=False)

print("Updated outputs with FDR columns and winsorization check.")
print("\\nAll outputs:")
print("  1. panel_country_year.xlsx")
print("  2. correlation_by_topic.xlsx (now with p_spearman_fdr)")
print("  3. alignment_scores.xlsx")
print("  4. comparison_program_vs_no_program.xlsx (now with p_mannwhitney_fdr + winsorization sheet)")

## 8. Charts

All charts saved to `charts/` folder.

### 8.1 Alignment Score Over Time

In [ ]:
# Alignment score trend over years, split by program status
yearly_all = align_df.groupby('year')['alignment_score'].agg(['mean', 'median', 'std', 'count']).reset_index()
yearly_pgm = align_df[align_df['during_program'] == 1].groupby('year')['alignment_score'].agg(['mean', 'median']).reset_index()
yearly_nopgm = align_df[align_df['during_program'] == 0].groupby('year')['alignment_score'].agg(['mean', 'median']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: overall trend with confidence band
axes[0].plot(yearly_all['year'], yearly_all['mean'], 'o-', color='#2c3e50', linewidth=2, label='Mean')
axes[0].fill_between(yearly_all['year'],
                     yearly_all['mean'] - yearly_all['std'],
                     yearly_all['mean'] + yearly_all['std'],
                     alpha=0.2, color='#2c3e50', label='±1 SD')
axes[0].plot(yearly_all['year'], yearly_all['median'], 's--', color='#e67e22', linewidth=1.5, label='Median')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Alignment Score')
axes[0].set_title('DSA↔Parent Alignment Over Time')
axes[0].legend()
axes[0].set_xlim(2004.5, 2024.5)

# Right: split by program status
axes[1].plot(yearly_pgm['year'], yearly_pgm['mean'], 'o-', color='#e74c3c', linewidth=2, label='In-Program (mean)')
axes[1].plot(yearly_nopgm['year'], yearly_nopgm['mean'], 's-', color='#3498db', linewidth=2, label='Not-In-Program (mean)')
axes[1].plot(yearly_pgm['year'], yearly_pgm['median'], 'o--', color='#e74c3c', linewidth=1, alpha=0.5, label='In-Program (median)')
axes[1].plot(yearly_nopgm['year'], yearly_nopgm['median'], 's--', color='#3498db', linewidth=1, alpha=0.5, label='Not-In-Program (median)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Alignment Score')
axes[1].set_title('Alignment: Program vs No-Program Over Time')
axes[1].legend(fontsize=8)
axes[1].set_xlim(2004.5, 2024.5)

plt.tight_layout()
plt.savefig('charts/correlations/alignment_time_series.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: charts/correlations/alignment_time_series.png")

### 8.2 Scatter Plots: Top Correlated Topics (DSA vs Parent)

In [ ]:
# Scatter plots for top 6 most correlated topics
top6 = corr_by_topic.dropna().head(6)['topic'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, topic in enumerate(top6):
    dsa_col = f'count_{topic}_per100w_dsa'
    parent_col = f'count_{topic}_per100w_parent'
    
    mask = df[dsa_col].notna() & df[parent_col].notna()
    x = df.loc[mask, parent_col]
    y = df.loc[mask, dsa_col]
    colors_sc = ['#e74c3c' if p == 1 else '#3498db' for p in df.loc[mask, 'during_program']]
    
    axes[i].scatter(x, y, c=colors_sc, alpha=0.4, s=20, edgecolor='none')
    
    # Add regression line
    z = np.polyfit(x, y, 1)
    p_line = np.poly1d(z)
    x_line = np.linspace(x.min(), x.max(), 100)
    axes[i].plot(x_line, p_line(x_line), 'k--', alpha=0.5, linewidth=1.5)
    
    r_val = corr_by_topic[corr_by_topic['topic'] == topic]['corr_spearman'].values[0]
    axes[i].set_title(f'{topic}\n(Spearman r = {r_val:.3f})', fontsize=11)
    axes[i].set_xlabel('Parent per100w')
    axes[i].set_ylabel('DSA per100w')

# Add legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0],[0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=8, label='In-Program'),
                   Line2D([0],[0], marker='o', color='w', markerfacecolor='#3498db', markersize=8, label='Not-In-Program')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('DSA vs Parent Document: Topic Intensity (per 100 words)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('charts/correlations/scatter_top_topics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: charts/correlations/scatter_top_topics.png")

### 8.3 Country × Topic Heatmap

In [ ]:
# Average per100w by country across DSAs — pick top 20 most variable topics
dsa_mentions = mentions[mentions['doc_type'] == 'DSA'].copy()

# Compute country means for per100w
country_topic = dsa_mentions.groupby('country')[per100w_cols].mean()

# Pick top 20 topics by cross-country variance (most interesting to visualize)
topic_variance = country_topic.var().sort_values(ascending=False)
top20_topics = topic_variance.head(20).index.tolist()

# Clean labels
labels = [c.replace('count_', '').replace('_per100w', '') for c in top20_topics]

fig, ax = plt.subplots(figsize=(16, 20))
sns.heatmap(country_topic[top20_topics].rename(columns=dict(zip(top20_topics, labels))),
            cmap='YlOrRd', ax=ax, linewidths=0.2, linecolor='white',
            xticklabels=True, yticklabels=True,
            cbar_kws={'label': 'Mean mentions per 100 words'})
ax.set_title('Country × Topic: Average DSA Mention Intensity (Top 20 Most Variable Topics)', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=9, rotation=45)

plt.tight_layout()
plt.savefig('charts/correlations/country_topic_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: charts/correlations/country_topic_heatmap.png")

### 8.4 Topic Prevalence Over Time

In [ ]:
# Select interesting topics to track over time (mix of economic and institutional)
track_topics = ['china', 'climate_shocks', 'debt_restructuring', 'domestic_debt',
                'health_shocks', 'political_social', 'non_concessional_borrowing',
                'arrears', 'world_bank', 'private_creditors']

# Compute yearly mean per100w for DSAs only
dsa_yearly = dsa_mentions.groupby('year')[[f'count_{t}_per100w' for t in track_topics]].mean()

fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# Split into two groups for readability
group1 = track_topics[:5]
group2 = track_topics[5:]

colors_palette = plt.cm.tab10(np.linspace(0, 1, 10))

for i, topic in enumerate(group1):
    col = f'count_{topic}_per100w'
    axes[0].plot(dsa_yearly.index, dsa_yearly[col], 'o-', linewidth=2,
                 label=topic, color=colors_palette[i], markersize=4)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Mean mentions per 100 words')
axes[0].set_title('Topic Prevalence in DSAs Over Time (Group 1)')
axes[0].legend(fontsize=9)
axes[0].set_xlim(2004.5, 2024.5)

for i, topic in enumerate(group2):
    col = f'count_{topic}_per100w'
    axes[1].plot(dsa_yearly.index, dsa_yearly[col], 'o-', linewidth=2,
                 label=topic, color=colors_palette[i+5], markersize=4)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Mean mentions per 100 words')
axes[1].set_title('Topic Prevalence in DSAs Over Time (Group 2)')
axes[1].legend(fontsize=9)
axes[1].set_xlim(2004.5, 2024.5)

plt.tight_layout()
plt.savefig('charts/correlations/topic_prevalence_time.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: charts/correlations/topic_prevalence_time.png")

### 8.5 Program Effect: Diverging Bar Chart

In [ ]:
# Diverging bar chart: all 68 topics, colored by significance after FDR
plot_data = sheet_a_fdr.sort_values('diff_means').copy()
plot_data['label'] = plot_data['topic']

fig, ax = plt.subplots(figsize=(12, 16))

colors_bar = []
for _, row in plot_data.iterrows():
    if row['sig_after_fdr']:
        colors_bar.append('#e74c3c' if row['diff_means'] > 0 else '#3498db')
    else:
        colors_bar.append('#d5d8dc')

ax.barh(range(len(plot_data)), plot_data['diff_means'], color=colors_bar, height=0.7, edgecolor='none')
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels(plot_data['label'], fontsize=8)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Difference in Mean per100w\n(In-Program − Not-In-Program)', fontsize=11)
ax.set_title('Program Effect on DSA Topic Mentions\n(colored = significant after FDR correction)', fontsize=13)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e74c3c', label='Significantly higher in program'),
    Patch(facecolor='#3498db', label='Significantly lower in program'),
    Patch(facecolor='#d5d8dc', label='Not significant after FDR'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('charts/correlations/program_effect_diverging.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: charts/correlations/program_effect_diverging.png")

In [ ]:
# Scatter plots — filtered: exclude pairs where BOTH DSA and parent have per100w == 0
top6 = corr_by_topic.dropna().head(6)['topic'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, topic in enumerate(top6):
    dsa_col = f'count_{topic}_per100w_dsa'
    parent_col = f'count_{topic}_per100w_parent'
    
    mask = df[dsa_col].notna() & df[parent_col].notna()
    sub = df.loc[mask].copy()
    
    # Count and exclude (0,0) pairs
    both_zero = (sub[dsa_col] == 0) & (sub[parent_col] == 0)
    n_excluded = both_zero.sum()
    sub = sub[~both_zero]
    
    x = sub[parent_col]
    y = sub[dsa_col]
    colors_sc = ['#e74c3c' if p == 1 else '#3498db' for p in sub['during_program']]
    
    axes[i].scatter(x, y, c=colors_sc, alpha=0.4, s=25, edgecolor='none')
    
    # Regression line
    if len(x) > 2:
        z = np.polyfit(x, y, 1)
        p_line = np.poly1d(z)
        x_line = np.linspace(x.min(), x.max(), 100)
        axes[i].plot(x_line, p_line(x_line), 'k--', alpha=0.5, linewidth=1.5)
    
    r_val = corr_by_topic[corr_by_topic['topic'] == topic]['corr_spearman'].values[0]
    axes[i].set_title(f'{topic}\n(Spearman r = {r_val:.3f})', fontsize=11)
    axes[i].set_xlabel('Parent per100w')
    axes[i].set_ylabel('DSA per100w')
    axes[i].annotate(f'{n_excluded} pairs with both=0 excluded',
                     xy=(0.02, 0.96), xycoords='axes fraction',
                     fontsize=7, color='grey', va='top')

from matplotlib.lines import Line2D
legend_elements = [Line2D([0],[0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=8, label='In-Program'),
                   Line2D([0],[0], marker='o', color='w', markerfacecolor='#3498db', markersize=8, label='Not-In-Program')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('DSA vs Parent Document: Topic Intensity — Excluding (0,0) Pairs', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('charts/correlations/scatter_top_topics_filtered.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: charts/correlations/scatter_top_topics_filtered.png")

In [ ]:
# Enhanced scatter: dual regression, 1:1 line, outlier labels, marginal distributions, CI bands
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

top6 = corr_by_topic.dropna().head(6)['topic'].tolist()

fig = plt.figure(figsize=(22, 15))
outer_grid = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

for idx, topic in enumerate(top6):
    dsa_col = f'count_{topic}_per100w_dsa'
    parent_col = f'count_{topic}_per100w_parent'

    mask = df[dsa_col].notna() & df[parent_col].notna()
    sub = df.loc[mask].copy()
    both_zero = (sub[dsa_col] == 0) & (sub[parent_col] == 0)
    n_excluded = both_zero.sum()
    sub = sub[~both_zero]

    x_all = sub[parent_col].values
    y_all = sub[dsa_col].values
    pgm = sub['during_program'].values
    countries = sub['country'].values

    # Inner grid: main scatter + marginals
    row, col = divmod(idx, 3)
    inner = outer_grid[row, col].subgridspec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
                                              hspace=0.05, wspace=0.05)
    ax_main = fig.add_subplot(inner[1, 0])
    ax_top = fig.add_subplot(inner[0, 0], sharex=ax_main)
    ax_right = fig.add_subplot(inner[1, 1], sharey=ax_main)

    # --- Main scatter ---
    for label, mask_pgm, color, marker in [('In-Program', pgm == 1, '#e74c3c', 'o'),
                                            ('Not-In-Program', pgm == 0, '#3498db', 's')]:
        ax_main.scatter(x_all[mask_pgm], y_all[mask_pgm], c=color, alpha=0.35, s=20,
                        edgecolor='none', marker=marker, label=label)

    # --- 1:1 line ---
    lim_max = max(x_all.max(), y_all.max()) * 1.05
    ax_main.plot([0, lim_max], [0, lim_max], 'k:', alpha=0.3, linewidth=1, label='1:1 line')

    # --- Dual regression lines with confidence bands ---
    for label, mask_pgm, color in [('In-Pgm', pgm == 1, '#e74c3c'),
                                    ('Not-In-Pgm', pgm == 0, '#3498db')]:
        xg = x_all[mask_pgm]
        yg = y_all[mask_pgm]
        if len(xg) < 10:
            continue
        # Fit
        z = np.polyfit(xg, yg, 1)
        p_fn = np.poly1d(z)
        x_line = np.linspace(xg.min(), xg.max(), 100)
        y_line = p_fn(x_line)
        ax_main.plot(x_line, y_line, color=color, linewidth=2, alpha=0.8)

        # Bootstrap CI band (quick 200 iterations)
        boot_lines = []
        rng = np.random.RandomState(42)
        for _ in range(200):
            idx_b = rng.choice(len(xg), len(xg), replace=True)
            zb = np.polyfit(xg[idx_b], yg[idx_b], 1)
            boot_lines.append(np.poly1d(zb)(x_line))
        boot_lines = np.array(boot_lines)
        ci_lo = np.percentile(boot_lines, 2.5, axis=0)
        ci_hi = np.percentile(boot_lines, 97.5, axis=0)
        ax_main.fill_between(x_line, ci_lo, ci_hi, color=color, alpha=0.1)

    # --- Outlier labels (top 5 by distance from regression line) ---
    z_all = np.polyfit(x_all, y_all, 1)
    predicted = np.poly1d(z_all)(x_all)
    residuals = np.abs(y_all - predicted)
    top_outliers = np.argsort(residuals)[-5:]
    for oi in top_outliers:
        ax_main.annotate(countries[oi], (x_all[oi], y_all[oi]),
                         fontsize=6, alpha=0.7, color='#2c3e50',
                         textcoords='offset points', xytext=(4, 4))

    r_val = corr_by_topic[corr_by_topic['topic'] == topic]['corr_spearman'].values[0]
    ax_main.set_xlabel('Parent per100w', fontsize=9)
    ax_main.set_ylabel('DSA per100w', fontsize=9)
    ax_main.annotate(f'{n_excluded} (0,0) pairs excluded', xy=(0.02, 0.97),
                     xycoords='axes fraction', fontsize=6, color='grey', va='top')

    # --- Marginal distributions ---
    ax_top.hist(x_all[pgm == 1], bins=30, alpha=0.5, color='#e74c3c', density=True, edgecolor='none')
    ax_top.hist(x_all[pgm == 0], bins=30, alpha=0.5, color='#3498db', density=True, edgecolor='none')
    ax_top.set_title(f'{topic} (r={r_val:.3f})', fontsize=11, fontweight='bold')
    ax_top.tick_params(labelbottom=False, labelleft=False, left=False)
    ax_top.spines[['top','right','left']].set_visible(False)

    ax_right.hist(y_all[pgm == 1], bins=30, alpha=0.5, color='#e74c3c', density=True,
                  orientation='horizontal', edgecolor='none')
    ax_right.hist(y_all[pgm == 0], bins=30, alpha=0.5, color='#3498db', density=True,
                  orientation='horizontal', edgecolor='none')
    ax_right.tick_params(labelbottom=False, labelleft=False, bottom=False)
    ax_right.spines[['top','right','bottom']].set_visible(False)

# Global legend
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=8, label='In-Program'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor='#3498db', markersize=8, label='Not-In-Program'),
    Line2D([0],[0], color='#e74c3c', linewidth=2, label='Regression (In-Pgm)'),
    Line2D([0],[0], color='#3498db', linewidth=2, label='Regression (Not-In-Pgm)'),
    Line2D([0],[0], color='black', linestyle=':', alpha=0.5, label='1:1 line (DSA = Parent)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=5, fontsize=10, bbox_to_anchor=(0.5, -0.01))

fig.suptitle('DSA vs Parent: Topic Intensity with Dual Regression, CI Bands & Outlier Labels',
             fontsize=15, y=1.01)
plt.savefig('charts/correlations/scatter_top_topics_enhanced.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: charts/correlations/scatter_top_topics_enhanced.png")

In [ ]:
# Enhanced scatter for selected topics: china, soes, data_transparency, domestic_debt, arrears, debt_restructuring
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

selected_topics = ['china', 'soes', 'debt_transparency', 'domestic_debt', 'arrears', 'debt_restructuring']

fig = plt.figure(figsize=(22, 15))
outer_grid = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

for idx, topic in enumerate(selected_topics):
    dsa_col = f'count_{topic}_per100w_dsa'
    parent_col = f'count_{topic}_per100w_parent'

    mask = df[dsa_col].notna() & df[parent_col].notna()
    sub = df.loc[mask].copy()
    both_zero = (sub[dsa_col] == 0) & (sub[parent_col] == 0)
    n_excluded = both_zero.sum()
    sub = sub[~both_zero]

    x_all = sub[parent_col].values
    y_all = sub[dsa_col].values
    pgm = sub['during_program'].values
    countries = sub['country'].values

    # Get Spearman r for this topic
    topic_row = corr_by_topic[corr_by_topic['topic'] == topic]
    r_val = topic_row['corr_spearman'].values[0] if len(topic_row) > 0 else np.nan

    row, col = divmod(idx, 3)
    inner = outer_grid[row, col].subgridspec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
                                              hspace=0.05, wspace=0.05)
    ax_main = fig.add_subplot(inner[1, 0])
    ax_top = fig.add_subplot(inner[0, 0], sharex=ax_main)
    ax_right = fig.add_subplot(inner[1, 1], sharey=ax_main)

    for label, mask_pgm, color, marker in [('In-Program', pgm == 1, '#e74c3c', 'o'),
                                            ('Not-In-Program', pgm == 0, '#3498db', 's')]:
        ax_main.scatter(x_all[mask_pgm], y_all[mask_pgm], c=color, alpha=0.35, s=20,
                        edgecolor='none', marker=marker, label=label)

    lim_max = max(x_all.max(), y_all.max()) * 1.05
    ax_main.plot([0, lim_max], [0, lim_max], 'k:', alpha=0.3, linewidth=1, label='1:1 line')

    for label, mask_pgm, color in [('In-Pgm', pgm == 1, '#e74c3c'),
                                    ('Not-In-Pgm', pgm == 0, '#3498db')]:
        xg = x_all[mask_pgm]
        yg = y_all[mask_pgm]
        if len(xg) < 10:
            continue
        z = np.polyfit(xg, yg, 1)
        p_fn = np.poly1d(z)
        x_line = np.linspace(xg.min(), xg.max(), 100)
        y_line = p_fn(x_line)
        ax_main.plot(x_line, y_line, color=color, linewidth=2, alpha=0.8)

        boot_lines = []
        rng = np.random.RandomState(42)
        for _ in range(200):
            idx_b = rng.choice(len(xg), len(xg), replace=True)
            zb = np.polyfit(xg[idx_b], yg[idx_b], 1)
            boot_lines.append(np.poly1d(zb)(x_line))
        boot_lines = np.array(boot_lines)
        ci_lo = np.percentile(boot_lines, 2.5, axis=0)
        ci_hi = np.percentile(boot_lines, 97.5, axis=0)
        ax_main.fill_between(x_line, ci_lo, ci_hi, color=color, alpha=0.1)

    z_all = np.polyfit(x_all, y_all, 1)
    predicted = np.poly1d(z_all)(x_all)
    residuals = np.abs(y_all - predicted)
    top_outliers = np.argsort(residuals)[-5:]
    for oi in top_outliers:
        ax_main.annotate(countries[oi], (x_all[oi], y_all[oi]),
                         fontsize=6, alpha=0.7, color='#2c3e50',
                         textcoords='offset points', xytext=(4, 4))

    ax_main.set_xlabel('Parent per100w', fontsize=9)
    ax_main.set_ylabel('DSA per100w', fontsize=9)
    ax_main.annotate(f'{n_excluded} (0,0) pairs excluded', xy=(0.02, 0.97),
                     xycoords='axes fraction', fontsize=6, color='grey', va='top')

    ax_top.hist(x_all[pgm == 1], bins=30, alpha=0.5, color='#e74c3c', density=True, edgecolor='none')
    ax_top.hist(x_all[pgm == 0], bins=30, alpha=0.5, color='#3498db', density=True, edgecolor='none')
    ax_top.set_title(f'{topic} (r={r_val:.3f})', fontsize=11, fontweight='bold')
    ax_top.tick_params(labelbottom=False, labelleft=False, left=False)
    ax_top.spines[['top','right','left']].set_visible(False)

    ax_right.hist(y_all[pgm == 1], bins=30, alpha=0.5, color='#e74c3c', density=True,
                  orientation='horizontal', edgecolor='none')
    ax_right.hist(y_all[pgm == 0], bins=30, alpha=0.5, color='#3498db', density=True,
                  orientation='horizontal', edgecolor='none')
    ax_right.tick_params(labelbottom=False, labelleft=False, bottom=False)
    ax_right.spines[['top','right','bottom']].set_visible(False)

legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=8, label='In-Program'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor='#3498db', markersize=8, label='Not-In-Program'),
    Line2D([0],[0], color='#e74c3c', linewidth=2, label='Regression (In-Pgm)'),
    Line2D([0],[0], color='#3498db', linewidth=2, label='Regression (Not-In-Pgm)'),
    Line2D([0],[0], color='black', linestyle=':', alpha=0.5, label='1:1 line (DSA = Parent)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=5, fontsize=10, bbox_to_anchor=(0.5, -0.01))

fig.suptitle('DSA vs Parent: Selected Topics — Dual Regression, CI Bands & Outlier Labels',
             fontsize=15, y=1.01)
plt.savefig('charts/correlations/scatter_selected_topics_enhanced.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: charts/correlations/scatter_selected_topics_enhanced.png")